# Data normalization
Use the z-score normalization. Also include ethnicity and gender processing.


In [ ]:
import pandas as pd
import os
from pathlib import Path
import numpy as np

In [ ]:
# load data
time_resolution = "1h"
impute_method = "bffill"
interested_split = 'test'

if impute_method == "bffill":
    imputed_dir = f"data/eICU_first24h_ts{time_resolution}/imputed"
    export_dir = f"data/eICU_first24h_ts{time_resolution}"
else:
    imputed_dir = f"data/eICU_first24h_ts{time_resolution}_impute-{impute_method}/imputed"
    export_dir = f"data/eICU_first24h_ts{time_resolution}_impute-{impute_method}"

Path(os.path.join(export_dir, interested_split)).mkdir(exist_ok=True, parents=True)

train_demo_path = os.path.join(imputed_dir, 'train', 'demographics.csv')
train_ts_path = os.path.join(imputed_dir, 'train', 'time-series.csv')

demo_path = os.path.join(imputed_dir, interested_split, 'demographics.csv')
ts_path = os.path.join(imputed_dir, interested_split, 'time-series.csv')
label_icumortality_path = os.path.join(imputed_dir, interested_split, 'label_icumortality.csv')
label_los_path = os.path.join(imputed_dir, interested_split, 'label_los.csv')
label_medication_path = os.path.join(imputed_dir, interested_split, 'label_medication.csv')
label_phenotype_path = os.path.join(imputed_dir, interested_split, 'label_phenotype.csv')

demo_export_path = os.path.join(export_dir, interested_split, 'demographics.csv')
ts_export_path = os.path.join(export_dir, interested_split, 'time-series.csv')
label_icumortality_export_path = os.path.join(export_dir, interested_split, 'label_icumortality.csv')
label_los_export_path = os.path.join(export_dir, interested_split, 'label_los.csv')
label_medication_export_path = os.path.join(export_dir, interested_split, 'label_medication.csv')
label_phenotype_export_path = os.path.join(export_dir, interested_split, 'label_phenotype.csv')

train_demo_df = pd.read_csv(train_demo_path)
train_ts_df = pd.read_csv(train_ts_path)

demo_df = pd.read_csv(demo_path)
ts_df = pd.read_csv(ts_path)
label_icumortality_df = pd.read_csv(label_icumortality_path)
label_los_df = pd.read_csv(label_los_path)
label_medication_df = pd.read_csv(label_medication_path)
label_phenotype_df = pd.read_csv(label_phenotype_path)

print(f"stay_id num of demo_df:{demo_df['stay_id'].unique().shape[0]}")
print(f"stay_id num of vital_df:{ts_df['stay_id'].unique().shape[0]}")
print(f"stay_id num of label_icumortality_df:{label_icumortality_df['stay_id'].unique().shape[0]}")
print(f"stay_id num of label_los_df:{label_los_df['stay_id'].unique().shape[0]}")
print(f"stay_id num of label_medication_df:{label_medication_df['stay_id'].unique().shape[0]}")
print(f"stay_id num of label_phenotype_df:{label_phenotype_df['stay_id'].unique().shape[0]}")

## Normalize

In [ ]:
# get the min, max, mean of each feature
demo_features = ['age']

train_demo_mean = train_demo_df.loc[:, demo_features].mean().values
train_demo_std = train_demo_df.loc[:, demo_features].std().values
train_ts_mean = train_ts_df.iloc[:, 3:].mean().values.reshape(1, -1)
train_ts_std = train_ts_df.iloc[:, 3:].std().values.reshape(1, -1)

train_demo_cv = train_demo_std / train_demo_mean
train_ts_cv = train_ts_std / train_ts_mean
print(f"train_demo_cv:{train_demo_cv}")
print(f"train_ts_cv:{train_ts_cv}")

In [ ]:
# apply the z-score normalization
demo_df.loc[:, demo_features] = ((demo_df.loc[:, demo_features] - train_demo_mean) / train_demo_std)
ts_df.iloc[:, 3:] = (ts_df.iloc[:, 3:] - train_ts_mean) / train_ts_std

demo_df.head()

In [ ]:
ts_df.head()

### Add subject_id into labels

In [ ]:
# export the processed data
demo_df.to_csv(demo_export_path, index=False)
ts_df.to_csv(ts_export_path, index=False)
label_icumortality_df.to_csv(label_icumortality_export_path, index=False)
label_los_df.to_csv(label_los_export_path, index=False)
label_medication_df.to_csv(label_medication_export_path, index=False)
label_phenotype_df.to_csv(label_phenotype_export_path, index=False)

print(f"Successfully export {interested_split} data to {export_dir}/{interested_split}/")